# Etapa 1. Python básico no Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/milho_oeste/blob/main/notebooks/01_python_basico.ipynb)

Este notebook ensina o mínimo de Python para seguir o projeto. Ele mistura **células de texto** (como esta) e
**células de código** (com fundo cinza).

**Como usar:**
- Clique numa célula de código e aperte **Shift + Enter** para rodá-la. O resultado aparece logo abaixo.
- Rode as células **em ordem**, de cima para baixo.
- Tudo o que vem depois de `#` numa linha é **comentário**: o Python ignora, serve para explicar o código.
- Antes de começar, faça **Arquivo → Salvar uma cópia no Drive** e trabalhe na cópia.
- Nos exercícios, escreva a resposta na célula vazia e rode. Errar faz parte; leia a última linha da mensagem
  de erro, que geralmente diz o que aconteceu.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# No Colab, esta célula:
#   1) instala as bibliotecas que o Colab não tem;
#   2) copia o projeto do GitHub (dados + funções) para a máquina do Colab;
#   3) conecta o seu Google Drive, onde os resultados ficam guardados entre uma sessão e outra.
# Fora do Colab (no computador do orientador), ela só acerta as pastas.
import os
import sys
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules        # True quando o notebook roda no Colab

if NO_COLAB:
    !pip -q install agrometeorologiapy==1.0.1
    if Path("/content/milho_oeste").exists():
        !git -C /content/milho_oeste pull -q     # já copiado nesta sessão: só atualiza
    else:
        !git clone -q https://github.com/fcoliveira-utfpr/milho_oeste.git /content/milho_oeste
    os.chdir("/content/milho_oeste")             # entra na pasta do projeto
    from google.colab import drive
    drive.mount("/content/drive")                # pede autorização na primeira vez
    SAIDA = Path("/content/drive/MyDrive/milho_oeste/resultados")
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
    SAIDA = Path("resultados")

SAIDA.mkdir(parents=True, exist_ok=True)         # cria a pasta de resultados, se não existir
sys.path.append("src")                           # permite importar as funções do projeto (pasta src/)
print("Pasta do projeto:", Path.cwd())
print("Resultados em   :", SAIDA)

## 1. Variáveis e contas

Uma **variável** é um nome que guarda um valor. O sinal `=` **guarda** o valor da direita no nome da esquerda
(não é "igual" da matemática). `print` mostra valores na tela.

In [ ]:
tmax = 31.2      # temperatura máxima do dia (°C)
tmin = 19.8      # temperatura mínima do dia (°C)

tmed = (tmax + tmin) / 2          # as contas seguem a ordem da matemática; use parênteses
print("Temperatura média:", tmed, "°C")

# f-string: um texto com f na frente aceita variáveis entre chaves; ":.1f" = uma casa decimal
print(f"Amplitude térmica: {tmax - tmin:.1f} °C")

## 2. Listas e o laço `for`

Uma **lista** guarda vários valores em ordem, entre colchetes. O laço `for` repete um bloco de código para cada
item. O bloco repetido é o que está **recuado** (4 espaços) abaixo do `for`.

In [ ]:
chuvas = [0.0, 12.4, 3.1, 0.0, 25.8, 7.2, 0.0]   # chuva (mm) de uma semana

print("Primeiro dia:", chuvas[0])     # em Python, a contagem começa em 0
print("Último dia:", chuvas[-1])      # -1 = último item
print("Número de dias:", len(chuvas))

total = 0
for chuva in chuvas:                  # "para cada chuva na lista chuvas"
    total = total + chuva             # acumula
print(f"Chuva da semana: {total:.1f} mm")
print(f"Conferindo com sum(): {sum(chuvas):.1f} mm")

## 3. Decisões com `if`

`if` executa um bloco **só se** a condição for verdadeira; `elif` e `else` tratam os outros casos.

In [ ]:
for tmin_dia in [8.5, 3.0, 1.2, -0.5]:
    if tmin_dia <= 0:
        situacao = "geada forte"
    elif tmin_dia <= 2:
        situacao = "risco de geada"       # o critério usado no projeto: Tmin ≤ 2 °C
    else:
        situacao = "sem risco"
    print(f"Tmin = {tmin_dia:5.1f} °C -> {situacao}")

## 4. Funções

Uma **função** é uma receita com nome: recebe valores (argumentos), faz contas e devolve um resultado com
`return`. Ela evita repetir o mesmo código. Abaixo, os **graus-dia** (GD), que medem o calor acumulado pela
planta acima da temperatura-base (Tb = 10 °C para o milho). É com a soma dos GD que o modelo decide em que fase
o milho está.

In [ ]:
def graus_dia(tmax, tmin, tb=10.0):
    """Graus-dia de um dia (°C dia). tb tem valor padrão 10 °C, então pode ser omitido."""
    tmed = (tmax + tmin) / 2
    return max(tmed - tb, 0.0)        # nunca negativo: abaixo de Tb a planta não se desenvolve

print(graus_dia(31.2, 19.8))          # usa tb = 10
print(graus_dia(31.2, 19.8, tb=8))    # muda tb só nesta chamada

dias = [(31.2, 19.8), (29.5, 18.1), (27.0, 15.3), (24.0, 9.0)]   # lista de pares (Tmax, Tmin)
soma = 0
for tx, tn in dias:                   # cada par é "desempacotado" em tx e tn
    soma = soma + graus_dia(tx, tn)
print(f"Graus-dia acumulados em {len(dias)} dias: {soma:.1f} °C dia")

## 5. Dicionários

Um **dicionário** guarda pares **chave: valor**, entre chaves `{}`. É como uma tabela de consulta. O projeto usa
um dicionário para os valores de **ky** de cada fase do milho (Doorenbos & Kassam, 1979).

In [ ]:
KY_DK = {"estabelecimento": 0.4, "vegetativo": 0.4, "floracao": 1.5, "enchimento": 0.5, "maturacao": 0.2}

print("ky da floração:", KY_DK["floracao"])        # consulta pela chave
for fase, ky in KY_DK.items():                      # .items() percorre os pares
    print(f"{fase:16s} ky = {ky}")

### Exercício 1

Escreva uma função `dias_quentes(lista_tmax)` que receba uma lista de temperaturas máximas e devolva **quantos
dias** tiveram Tmax ≥ 35 °C (o calor que prejudica o florescimento do milho). Teste com
`[33.0, 35.5, 36.1, 31.0, 35.0]` (resposta: 3).

In [ ]:
# escreva sua resposta aqui

## 6. Tabelas com pandas

O **pandas** é a biblioteca de tabelas do Python. Uma tabela (*DataFrame*) funciona como uma planilha: cada
coluna tem um nome. Vamos abrir o clima diário dos 50 municípios do Oeste (base de Xavier et al.).

Colunas: `pr` chuva (mm), `Tmax` e `Tmin` (°C), `RH` umidade relativa (%), `Rs` radiação solar
(MJ m⁻² dia⁻¹), `u2` vento a 2 m (m s⁻¹) e `ET` a evapotranspiração de referência (ETo, mm dia⁻¹) já calculada
pela base.

In [ ]:
import pandas as pd

clima = pd.read_parquet("dados/clima_oeste.parquet")      # lê o arquivo de clima
mun = pd.read_csv("dados/municipios_oeste.csv")            # lê a tabela dos municípios

print(clima.shape)          # (número de linhas, número de colunas)
clima.head()                # mostra as 5 primeiras linhas

In [ ]:
# Selecionar linhas: escreva a condição entre colchetes.
sh = mun[mun["municipio"] == "Santa Helena"].iloc[0]       # .iloc[0] = a primeira (e única) linha
print(sh)

clima_sh = clima[clima["cod_ibge"] == sh["cod_ibge"]]     # só o clima de Santa Helena
clima_sh = clima_sh.set_index("data")                     # a data vira o "índice" (rótulo das linhas)
clima_sh.loc["2020-03-01":"2020-03-07"]                   # .loc seleciona pelo rótulo: uma semana

## 7. A ETo de um dia, passo a passo, com a biblioteca

A **evapotranspiração de referência** (ETo) é a água que uma grama bem irrigada perderia num dia. Pelo método de
Penman-Monteith (FAO 56; Allen et al., 1998), ela precisa de várias grandezas intermediárias. A biblioteca
`agrometeorologiapy` tem uma função para cada uma. Use `help(função)` para ler a documentação, em português.

In [ ]:
import agrometeorologiapy as amp
help(amp.eto_penman_monteith_fao56)

In [ ]:
dia = clima_sh.loc["2020-03-01"]          # uma linha da tabela (um dia)
lat, alt = sh["latitude"], sh["altitude_m"]
tmed = (dia["Tmax"] + dia["Tmin"]) / 2

# Pressões de vapor (kPa): de saturação (es) e atual (ea)
es = (amp.es_tetens(dia["Tmax"]) + amp.es_tetens(dia["Tmin"])) / 2
ea = amp.ea_umidade(es, dia["RH"])

# Declive da curva de pressão de vapor (Delta) e constante psicrométrica (gamma)
delta = amp.declive_pressao_vapor(tmed)
gamma = amp.constante_psicrometrica(amp.patm_altitude(alt))

# Radiação: extraterrestre (Qo), de céu claro (Qg_cs), balanços de ondas curtas e longas e saldo (Rn)
nda = dia.name.day_of_year                                 # dia do ano (1 a 366)
dec = amp.declinacao_solar(nda)
qo = amp.irradiancia_extraterrestre(lat, dec, amp.angulo_horario_nascer(lat, dec), amp.fator_correcao_distancia(nda))
qg_cs = (0.75 + 2e-5 * alt) * qo
boc = amp.boc_saldo(dia["Rs"], r=0.23)                    # albedo da grama de referência = 0,23
bol = amp.bol_saldo(dia["Tmax"], dia["Tmin"], ea, dia["Rs"], qg_cs)
rn = amp.saldo_radiacao(boc, bol)

eto = amp.eto_penman_monteith_fao56(rn, 0.0, tmed, dia["u2"], es, ea, delta, gamma)   # G = 0 na escala diária
print(f"Rn = {rn:.2f} MJ/m² dia")
print(f"ETo calculada = {eto:.2f} mm/dia   |   ETo da base Xavier = {dia['ET']:.2f} mm/dia")

## 8. A mesma conta para um mês inteiro, e um gráfico

Em vez de repetir a conta dia a dia, as funções aceitam **colunas inteiras** de uma vez. O **matplotlib** faz os
gráficos.

In [ ]:
import matplotlib.pyplot as plt

m = clima_sh.loc["2020-03-01":"2020-03-31"].copy()        # um mês; .copy() evita avisos ao criar colunas
tm = (m["Tmax"] + m["Tmin"]) / 2
es = (amp.es_tetens(m["Tmax"]) + amp.es_tetens(m["Tmin"])) / 2
ea = amp.ea_umidade(es, m["RH"])
nda = m.index.day_of_year.to_numpy()
dec = amp.declinacao_solar(nda)
qo = amp.irradiancia_extraterrestre(lat, dec, amp.angulo_horario_nascer(lat, dec), amp.fator_correcao_distancia(nda))
rn = amp.saldo_radiacao(amp.boc_saldo(m["Rs"], r=0.23),
                        amp.bol_saldo(m["Tmax"], m["Tmin"], ea, m["Rs"], (0.75 + 2e-5 * alt) * qo))
m["ETo_calc"] = amp.eto_penman_monteith_fao56(rn, 0.0, tm, m["u2"], es, ea, amp.declive_pressao_vapor(tm),
                                              amp.constante_psicrometrica(amp.patm_altitude(alt)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(m.index, m["ETo_calc"], marker="o", label="ETo calculada (agrometeorologiapy)")
ax.plot(m.index, m["ET"], marker="s", label="ETo da base Xavier")
ax.set_ylabel("ETo (mm/dia)")
ax.set_title("Santa Helena, março de 2020")
ax.legend()
plt.show()

### Exercício 2

1. Quanto choveu em Santa Helena em março de 2020? E a ETo acumulada no mês? O balanço (chuva − ETo) foi positivo
   ou negativo? (dica: `m["pr"].sum()`)
2. Repita o gráfico para março de 2019. O que mudou?

### O que anotar

Escreva numa célula de texto, com suas palavras: o que é a ETo, de quais variáveis ela depende e por que a ETo
calculada não é idêntica à da base (pense nas aproximações usadas).

In [ ]:
# escreva sua resposta aqui